In [ ]:
from pathlib import Path

import polars as pl
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from forecasting.evaluation.backtest import prepare_training_at_cutoff
from forecasting.evaluation.benchmark import (
    cutoffs_to_datetimes,
    get_validation_data,
    load_benchmark_config,
    load_benchmark_data,
    parse_benchmark_config,
)

In [ ]:
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

config_path = project_root / "configs" / "benchmark_bike_validation.yaml"

config = parse_benchmark_config(load_benchmark_config(config_path))
validation = config.validation

data = load_benchmark_data(project_root, config.input_path)
data = get_validation_data(data, validation.train_start, validation.test_start)

first_cutoff = cutoffs_to_datetimes(validation.cutoffs)[0]
history = data.filter(pl.col("timestamp") <= first_cutoff)

report, train = prepare_training_at_cutoff(
    history,
    first_cutoff,
    initial_train_days=validation.initial_train_days,
)

print("First cutoff:", first_cutoff)
print("Eligible series:", train["series_id"].n_unique())
print("Training rows:", train.height)

In [ ]:
series_summary = (
    train.group_by("series_id")
    .agg(
        n_days=pl.len(),
        mean_count=pl.col("y").mean(),
        std_count=pl.col("y").std(),
        imputed_days=pl.col("is_imputed").sum(),
    )
    .sort(["mean_count", "series_id"])
)

series_summary

In [ ]:
series_id = series_summary["series_id"][series_summary.height // 2]

series = (
    train.filter(pl.col("series_id") == series_id)
    .sort("timestamp")
)

print("Selected series:", series_id)
print("Imputed days:", series["is_imputed"].sum())

In [ ]:
import matplotlib.pyplot as plt

values = series["y"].to_numpy()
dates = series["timestamp"].to_list()

fig, axes = plt.subplots(3, 1, figsize=(12, 10))

axes[0].plot(dates, values)
axes[0].set_title(f"Daily counts — {series_id}")

plot_acf(values, lags=28, alpha=0.05, ax=axes[1])
axes[1].set_title("ACF — original series")

plot_pacf(values, lags=28, alpha=0.05, method="ywm", ax=axes[2])
axes[2].set_title("PACF — original series")

plt.tight_layout()
plt.show()

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss

adf_result = adfuller(values, autolag="AIC")
kpss_result = kpss(values, regression="c", nlags="auto")

print("ADF statistic:", adf_result[0])
print("ADF p-value:", adf_result[1])
print("KPSS statistic:", kpss_result[0])
print("KPSS p-value:", kpss_result[1])

### Analyse de la série originale

La série étudiée correspond au compteur `100056046-101056046`, choisi au
milieu du classement selon la fréquentation moyenne. Elle contient
180 jours disponibles jusqu'au 23 mars 2026.

L'ACF présente des pics aux retards de 7, 14, 21 et 28 jours : cela suggère
une dépendance hebdomadaire. La série montre aussi une baisse temporaire
autour de fin décembre et début janvier, dont la cause n'est pas établie.

#### Tests de stationnarité

Au seuil de 5 % :

| Test | p-value | Interprétation |
|---|---:|---|
| ADF | 0,079 | On ne rejette pas l'hypothèse de racine unitaire. |
| KPSS | 0,058 | On ne rejette pas l'hypothèse de stationnarité autour d'un niveau constant. |

Ces résultats ne permettent pas de conclure avec certitude : les deux
p-values sont proches du seuil et les tests examinent des hypothèses
différentes. La saisonnalité hebdomadaire peut compliquer leur
interprétation.

Une p-value n'est pas la probabilité que la série soit stationnaire.
Nous comparerons ensuite une différenciation quotidienne et une
différenciation hebdomadaire, en examinant aussi leurs graphiques ACF et PACF.

In [ ]:
import numpy as np

daily_diff = np.diff(values)
seasonal_diff = values[7:] - values[:-7]

for name, current_values in {
    "original": values,
    "daily difference": daily_diff,
    "weekly difference": seasonal_diff,
}.items():
    adf = adfuller(current_values, autolag="AIC", result_object=False)
    kpss_test = kpss(
        current_values,
        regression="c",
        nlags="auto",
        result_object=False,
    )

    print(f"{name}:")
    print(f"  Observations: {len(current_values)}")
    print(f"  ADF p-value: {adf[1]:.4f}")
    print(f"  KPSS p-value: {kpss_test[1]:.4f}")

### Différenciation et tests de stationnarité

La différenciation transforme la série en calculant des écarts :

- **Quotidienne** : chaque jour est comparé au jour précédent.
- **Hebdomadaire** : chaque jour est comparé au même jour de la semaine précédente.

#### Comprendre les tests

Le test **ADF** (*Augmented Dickey-Fuller*) examine l'hypothèse
d'une racine unitaire. Une p-value inférieure à 0,05 conduit à rejeter
cette hypothèse ; cela ne prouve pas à lui seul que la série est stationnaire.

Le test **KPSS** examine ici l'hypothèse que la série est stationnaire
autour d'un niveau constant. Une p-value inférieure à 0,05 conduit à
rejeter cette hypothèse.

La p-value mesure à quel point le résultat observé est compatible avec
l'hypothèse testée. Elle n'est pas la probabilité que cette hypothèse soit vraie.

#### Résultats au seuil de 5 %

| Série | Observations | p-value ADF | p-value KPSS |
|---|---:|---:|---:|
| Originale | 180 | 0,0789 | 0,0583 |
| Différence quotidienne | 179 | 0,0024 | > 0,10 |
| Différence hebdomadaire | 173 | 0,0143 | > 0,10 |

Pour les deux séries différenciées, l'ADF rejette l'hypothèse de racine
unitaire et le KPSS ne rejette pas son hypothèse de stationnarité. Ces
résultats sont compatibles avec la stationnarité.

Statsmodels affiche `0,10` comme limite de sa table KPSS et avertit que
les p-values réelles sont supérieures à cette valeur.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

for row, (name, diff_values, diff_dates) in enumerate(
    [
        ("Daily difference", daily_diff, dates[1:]),
        ("Weekly difference", seasonal_diff, dates[7:]),
    ]
):
    axes[row, 0].plot(diff_dates, diff_values)
    axes[row, 0].set_title(name)

    plot_acf(diff_values, lags=28, alpha=0.05, ax=axes[row, 1])
    axes[row, 1].set_title(f"ACF — {name}")

    plot_pacf(diff_values, lags=28, alpha=0.05, method="ywm", ax=axes[row, 2])
    axes[row, 2].set_title(f"PACF — {name}")

plt.tight_layout()
plt.show()

In [ ]:
import warnings

from statsmodels.tools.sm_exceptions import ConvergenceWarning
from statsmodels.tsa.statespace.sarimax import SARIMAX

orders = {
    "AR1": (1, 0, 0),
    "ARMA11": (1, 0, 1),
}

fitted_models = {}

for name, order in orders.items():
    model = SARIMAX(
        values,
        order=order,
        seasonal_order=(1, 0, 0, 7),
        trend="c",
    )

    with warnings.catch_warnings():
        warnings.simplefilter("error", ConvergenceWarning)
        result = model.fit(disp=False, maxiter=200)

    if not result.mle_retvals.get("converged", False):
        raise RuntimeError(f"{name} did not converge")

    fitted_models[name] = result
    print(f"{name}: AIC = {result.aic:.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, (name, result) in zip(axes, fitted_models.items()):
    residuals = np.asarray(result.resid)[result.loglikelihood_burn:]

    plot_acf(residuals, lags=28, alpha=0.05, ax=ax)
    ax.set_title(f"Residual ACF — {name}")

plt.tight_layout()
plt.show()

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox

model_degrees = {
    "AR1": 2,      # p=1 et P=1
    "ARMA11": 3,   # p=1, q=1 et P=1
}

for name, result in fitted_models.items():
    residuals = np.asarray(result.resid)[result.loglikelihood_burn:]

    test = acorr_ljungbox(
        residuals,
        lags=[7, 14, 28],
        model_df=model_degrees[name],
        return_df=True,
    )

    print(name)
    print(test)

In [ ]:
seasonal_model = SARIMAX(
    values,
    order=(1, 0, 0),
    seasonal_order=(1, 0, 1, 7),
    trend="c",
)

with warnings.catch_warnings():
    warnings.simplefilter("error", ConvergenceWarning)
    seasonal_result = seasonal_model.fit(disp=False, maxiter=200)

if not seasonal_result.mle_retvals.get("converged", False):
    raise RuntimeError("Seasonal MA model did not converge")

print(f"Seasonal MA model: AIC = {seasonal_result.aic:.2f}")

residuals = np.asarray(seasonal_result.resid)[
    seasonal_result.loglikelihood_burn:
]

print(
    acorr_ljungbox(
        residuals,
        lags=[7, 14, 28],
        model_df=3,  # p=1, P=1 et Q=1
        return_df=True,
    )
)

In [ ]:
differenced_model = SARIMAX(
    values,
    order=(1, 0, 0),
    seasonal_order=(1, 1, 1, 7),
    trend="n",  # Sans constante
)

with warnings.catch_warnings():
    warnings.simplefilter("error", ConvergenceWarning)
    differenced_result = differenced_model.fit(disp=False, maxiter=200)

if not differenced_result.mle_retvals.get("converged", False):
    raise RuntimeError("Differenced model did not converge")

residuals = np.asarray(differenced_result.resid)[
    differenced_result.loglikelihood_burn:
]

print(
    acorr_ljungbox(
        residuals,
        lags=[7, 14, 28],
        model_df=3,  # p=1, P=1 et Q=1
        return_df=True,
    )
)

plot_acf(residuals, lags=28, alpha=0.05)
plt.title("Residual ACF — seasonal difference")
plt.show()

In [ ]:
from datetime import timedelta

horizon = validation.horizon_days
forecast_dates = [
    first_cutoff + timedelta(days=day)
    for day in range(1, horizon + 1)
]

actuals = (
    pl.DataFrame({"timestamp": forecast_dates})
    .join(
        data.filter(pl.col("series_id") == series_id).select("timestamp", "y"),
        on="timestamp",
        how="left",
    )
    .sort("timestamp")
)

models_to_compare = {
    "Initial": fitted_models["AR1"],
    "Seasonal Q=1": seasonal_result,
    "Weekly difference": differenced_result,
}

predictions = {
    name: np.asarray(result.forecast(steps=horizon), dtype=np.float64)
    for name, result in models_to_compare.items()
}

print("Observed days:", actuals["y"].is_not_null().sum(), "/", horizon)

plt.figure(figsize=(12, 5))
plt.plot(actuals["timestamp"].to_list(), actuals["y"].to_list(), "ko", label="Observed")

for name, forecast in predictions.items():
    plt.plot(forecast_dates, forecast, label=name)

plt.title(f"Validation forecasts — {series_id}")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
from forecasting.evaluation.metrics import mase, wape

if actuals["y"].null_count() > 0:
    raise ValueError("Some observations are missing")

y_true = actuals["y"].to_numpy()

for name, forecast in predictions.items():
    model_wape = wape(y_true, forecast)
    model_mase = mase(y_true, forecast, values, season_length=7)

    print(f"{name}: WAPE={model_wape:.4f}, MASE={model_mase:.4f}")

In [ ]:
from dataclasses import replace

from forecasting.evaluation.benchmark import run_models_benchmark
from forecasting.models.sarima import SeasonalARIMA

first_validation = replace(
    validation,
    last_cutoff=validation.first_cutoff,
    cutoffs=(validation.first_cutoff,),
)

models = {
    "initial": SeasonalARIMA,
    "seasonal_q1": lambda: SeasonalARIMA(
        seasonal_order=(1, 0, 1, 7)
    ),
}

results = run_models_benchmark(data, first_validation, models)

for name, result in results.items():
    print(f"\n{name}")
    print("Series:", result.forecasts["series_id"].n_unique())
    print("Mean WAPE:", result.wape_scores["wape"].mean())
    print("Mean MASE:", result.scaled_scores["mase"].mean())

In [ ]:
# Keep the first-cutoff results
first_results = results

middle_cutoff = validation.cutoffs[len(validation.cutoffs) // 2]

middle_validation = replace(
    validation,
    first_cutoff=middle_cutoff,
    last_cutoff=middle_cutoff,
    cutoffs=(middle_cutoff,),
)

print("Middle cutoff:", middle_cutoff)

results = run_models_benchmark(data, middle_validation, models)

In [ ]:
for name, result in results.items():
    print(f"\n{name}")
    print("WAPE statuses:")
    print(result.wape_scores.group_by("wape_status").len())
    print("MASE statuses:")
    print(result.scaled_scores.group_by("mase_status").len())
    print(
        "Complete horizons:",
        result.wape_scores.filter(
            pl.col("n_evaluable") == validation.horizon_days
        ).height,
    )

comparison = (
    results["initial"].scaled_scores
    .select("series_id", "cutoff", pl.col("mase").alias("initial"))
    .join(
        results["seasonal_q1"].scaled_scores.select(
            "series_id", "cutoff", pl.col("mase").alias("q1")
        ),
        on=["series_id", "cutoff"],
    )
    .drop_nulls(["initial", "q1"])
)

print("\nComparable series:", comparison.height)
print("Q=1 better:", comparison.filter(pl.col("q1") < pl.col("initial")).height)
print("Initial better:", comparison.filter(pl.col("initial") < pl.col("q1")).height)
print("Equal:", comparison.filter(pl.col("q1") == pl.col("initial")).height)

In [ ]:
d_models = {
    "q1_d0": lambda: SeasonalARIMA(seasonal_order=(1, 0, 1, 7)),
    "q1_d1": lambda: SeasonalARIMA(seasonal_order=(1, 1, 1, 7)),
}

d_results = run_models_benchmark(data, first_validation, d_models)

for name, result in d_results.items():
    print(f"\n{name}")
    print("Series:", result.forecasts["series_id"].n_unique())
    print("Mean WAPE:", result.wape_scores["wape"].mean())
    print("Mean MASE:", result.scaled_scores["mase"].mean())
    print(result.scaled_scores.group_by("mase_status").len())

In [ ]:
d_comparison = (
    d_results["q1_d0"].scaled_scores
    .select("series_id", "cutoff", pl.col("mase").alias("d0"))
    .join(
        d_results["q1_d1"].scaled_scores.select(
            "series_id", "cutoff", pl.col("mase").alias("d1")
        ),
        on=["series_id", "cutoff"],
    )
    .drop_nulls(["d0", "d1"])
)

print("Comparable:", d_comparison.height)
print("D=1 better:", d_comparison.filter(pl.col("d1") < pl.col("d0")).height)
print("D=0 better:", d_comparison.filter(pl.col("d0") < pl.col("d1")).height)
print("Equal:", d_comparison.filter(pl.col("d1") == pl.col("d0")).height)

In [ ]:
with_constant = SARIMAX(
    values,
    order=(1, 0, 0),
    seasonal_order=(1, 1, 1, 7),
    trend="c",
)

with warnings.catch_warnings():
    warnings.simplefilter("error", ConvergenceWarning)
    with_constant_result = with_constant.fit(disp=False, maxiter=200)

if not with_constant_result.mle_retvals.get("converged", False):
    raise RuntimeError("Model with constant did not converge")

if actuals["y"].null_count() > 0:
    raise ValueError("Some observations are missing")

y_true = actuals["y"].to_numpy()

for name, result in {
    "Without constant": differenced_result,
    "With constant": with_constant_result,
}.items():
    forecast = np.asarray(result.forecast(steps=horizon), dtype=np.float64)
    print(
        f"{name}: WAPE={wape(y_true, forecast):.4f}, "
        f"MASE={mase(y_true, forecast, values, season_length=7):.4f}"
    )

In [ ]:
d_middle_results = run_models_benchmark(data, middle_validation, d_models)

for name, result in d_middle_results.items():
    print(f"\n{name}")
    print("Series:", result.forecasts["series_id"].n_unique())
    print("Mean WAPE:", result.wape_scores["wape"].mean())
    print("Mean MASE:", result.scaled_scores["mase"].mean())
    print(result.scaled_scores.group_by("mase_status").len())
    print(
        "Complete horizons:",
        result.wape_scores.filter(
            pl.col("n_evaluable") == validation.horizon_days
        ).height,
    )